In [7]:
import json
from pathlib import Path

import jieba
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer

N_TOPICS = 8
N_WORDS = 10
OUTPUT_DIR = Path("outputs/a_kmeans")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [8]:
with open("data/XiZhan/xizhan_analysis.json", "r", encoding="utf-8") as f:
    rawjs = json.load(f)

raw = pd.DataFrame(rawjs).rename(columns={"description": "content"})
documents = raw["content"].tolist()

In [9]:
if len(documents) < N_TOPICS:
    raise ValueError(f"可聚类文本仅有 {len(documents)} 条，少于 N_TOPICS={N_TOPICS}")

stop_words = set()
stopword_path = Path("stopword.txt")
if stopword_path.is_file():
    stop_words.update(
        word.strip()
        for word in stopword_path.read_text(encoding="utf-8").splitlines()
        if word.strip()
    )

def tokenize_zh(text: str) -> list[str]:
    return [
        word.strip()
        for word in jieba.lcut(str(text))
        if len(word.strip()) > 1
        and not word.strip().isdigit()
        and word.strip() not in stop_words
    ]

model = SentenceTransformer(
    "/Users/clsljt/Documents/hhgh/station/embModel/qwen3-emb-0.6b",
    device="mps",
    trust_remote_code=True,
)

embeddings = model.encode(
    documents,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True,
)

vectorizer_model = CountVectorizer(
    tokenizer=tokenize_zh,
    token_pattern=None,
    lowercase=False,
    min_df=1,
)
cluster_model = KMeans(n_clusters=N_TOPICS, random_state=42, n_init=10)

topic_model = BERTopic(
    embedding_model=None,
    hdbscan_model=cluster_model,
    vectorizer_model=vectorizer_model,
    top_n_words=30,
    low_memory=True,
)
topics, probs = topic_model.fit_transform(documents, embeddings)

topic_info = topic_model.get_topic_info()
topic_info.to_csv(OUTPUT_DIR / "kmeans_topic_info.csv", index=False, encoding="utf-8-sig")
print(f"已完成 {N_TOPICS} 个 KMeans 主题；主题信息保存到：{(OUTPUT_DIR / 'kmeans_topic_info.csv').resolve()}")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/4y/vlf2jr0n0bgc5_572sb76tzw0000gn/T/jieba.cache
Loading model cost 0.368 seconds.
Prefix dict has been built successfully.


已完成 8 个 KMeans 主题；主题信息保存到：/Users/clsljt/Documents/hhgh/station/outputs/a_kmeans/kmeans_topic_info.csv


In [13]:
valid_topics = [
    int(topic)
    for topic in topic_model.get_topic_info()["Topic"].tolist()
    if int(topic) != -1
]
if not valid_topics:
    raise ValueError("没有可展示的主题，请先确认 KMeans 聚类是否成功。")

keyword_rows = []
for topic_id in valid_topics:
    for rank, (keyword, score) in enumerate((topic_model.get_topic(topic_id) or [])[:N_WORDS], start=1):
        keyword_rows.append(
            {"topic": topic_id, "rank": rank, "keyword": keyword, "score": score}
        )
keyword_df = pd.DataFrame(keyword_rows)
keyword_csv_path = OUTPUT_DIR / "kmeans_barchart_keywords.csv"
keyword_df.to_csv(keyword_csv_path, index=False, encoding="utf-8-sig")

barchart = topic_model.visualize_barchart(
    topics=valid_topics,
    n_words=N_WORDS,
    title="KMeans 主题关键词权重（BERTopic 原生 barchart）",
    width=1200,
    height=max(450, 260 * ((len(valid_topics) + 3) // 4)),
)
barchart_path = OUTPUT_DIR / "kmeans_native_barchart.html"
barchart.write_html(barchart_path, include_plotlyjs=True)

# single_barchart_dir = OUTPUT_DIR / "individual_barcharts"
# single_barchart_dir.mkdir(exist_ok=True)
# single_barchart_paths = []
# for topic_id in valid_topics:
#     single_barchart = topic_model.visualize_barchart(
#         topics=[topic_id],
#         n_words=N_WORDS,
#         title=f"KMeans 主题 {topic_id} 关键词权重",
#         width=900,
#         height=420,
#     )
#     single_path = single_barchart_dir / f"barchart_topic_{topic_id}.html"
#     single_barchart.write_html(single_path, include_plotlyjs=True)
#     single_barchart_paths.append(single_path)

# print(f"原生 barchart 已保存到：{barchart_path.resolve()}")
# print(f"关键词明细已保存到：{keyword_csv_path.resolve()}")
# print(f"单主题 barchart 已保存 {len(single_barchart_paths)} 个到：{single_barchart_dir.resolve()}")
barchart